# 05_02_Saneamento_e_Winsorizacao

Este notebook executa o **Saneamento Estatístico de Outliers** das séries acionárias diárias.
A metodologia de winsorização robusta emprega o Z-score modificado de Iglewicz e Hoaglin (1993) sob a restrição metodológica de cômputo da Mediana e MAD **exclusivamente sobre variações não-nulas**, evitando a atenuação artificial da escala provocada pelo forward-fill em ativos de liquidez restrita.

**SRP (Single Responsibility Principle):** Este notebook responde apenas pela profilaxia de outliers nos retornos acionários e auditoria de qualidade corporativa.

## 1. Configurações, Parâmetros e Caminhos

In [1]:
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
from utils.config_loader import carregar_parametros
from utils.winsorizacao import winsorizar_painel_log
from utils.auditoria import (
    diagnosticar_retornos_impossiveis,
    gerar_log_winsorizacao,
    exportar_verificacao_corporativa
)

warnings.filterwarnings("ignore")

# Carregar parâmetros metodológicos
config = carregar_parametros("config.json")
K_MAD = config["K_MAD"]
C_MAD = config["C_MAD"]
MAD_SOBRE_NAO_NULOS = config["MAD_SOBRE_NAO_NULOS"]
LIMITE_IMPOSSIVEL = config["LIMITE_IMPOSSIVEL"]

# Resolução de caminhos
parent_path = Path.cwd().parent.parent
DIR_PAINEL = parent_path / "data" / "Painel_Dados"
DIR_RETORNOS = parent_path / "data" / "Retornos"

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")
print(f"[OK] Multiplicador robusto (k): {K_MAD}")
print(f"[OK] Fator de escala Gaussiano (c): {C_MAD}")
print(f"[OK] Estimativa sobre retornos não-nulos: {MAD_SOBRE_NAO_NULOS}")
print(f"[OK] Limite impossível: {LIMITE_IMPOSSIVEL*100:.0f}% a.d.")

[OK] Multiplicador robusto (k): 3.5
[OK] Fator de escala Gaussiano (c): 0.6745
[OK] Estimativa sobre retornos não-nulos: True
[OK] Limite impossível: 100% a.d.


## 2. Carregamento das Séries Temporais

In [2]:
ret_simples = pd.read_parquet(DIR_RETORNOS / "retornos_simples.parquet")
ret_log = pd.read_parquet(DIR_RETORNOS / "retornos_log.parquet")
painel_precos = pd.read_parquet(DIR_PAINEL / "painel_alinhado.parquet")

print(f"Retornos de Ações: {ret_simples.shape[0]} pregões × {ret_simples.shape[1]} ativos")

Retornos de Ações: 3966 pregões × 118 ativos


## 3. Diagnóstico Pré-Winsorização: Variações Economicamente Impossíveis (|R| > 100%)

In [3]:
audit = diagnosticar_retornos_impossiveis(ret_simples, painel_precos, limiar=LIMITE_IMPOSSIVEL)
print("=== Ativos com variações diárias brutas > 100% ===")
if len(audit) > 0:
    print(audit.to_string())
else:
    print("Nenhum ativo detectado com estouro de barreira.")

=== Ativos com variações diárias brutas > 100% ===
             dias_|R|>100%  maior_|R|_%  preco_mediano_R$
ACAO_FICT3               2   142.000000          0.627121
ACAO_LUPA3               2   148.000000          3.528903
ACAO_GOLL54              2 1,847.000000          0.147976
ACAO_TELB4               2   223.000000         22.509628
ACAO_AMER3               1   180.000000      1,616.798035


## 4. Execução da Winsorização Robusta via MAD Condicional

In [4]:
cols_acoes = ret_log.columns.tolist()

# Executa o clipping robusto nos log-retornos
ret_log_san, relatorio = winsorizar_painel_log(
    ret_log,
    k=K_MAD,
    c=C_MAD,
    nao_nulos=MAD_SOBRE_NAO_NULOS
)

# Reconverte as matrizes higienizadas para retornos simples
ret_simples_san = np.expm1(ret_log_san)

# Grava e carrega o log comparativo de truncamentos
rel = gerar_log_winsorizacao(relatorio, DIR_RETORNOS / "log_winsorizacao.csv")

n_v2 = int(rel["n_winsor_v2"].sum())
n_old = int(rel["n_winsor_com_zeros"].sum())

print(f"\n[+] Total de observações winsorizadas (v2): {n_v2} (corte condicional a não-nulos)")
print(f"    Referência incondicional (com zeros):   {n_old} (induz à sobre-winsorização)")
print(f"    Ganho de preservação de variação real:  {n_old - n_v2} observações mantidas.")

print("\n=== Ativos com Maior Preservação de Volatilidade Real (Diferença de Cortes) ===")
rel["reducao"] = rel["n_winsor_com_zeros"] - rel["n_winsor_v2"]
print(rel.sort_values("reducao", ascending=False).head(8)
        [["n_winsor_com_zeros","n_winsor_v2","reducao"]].to_string())


[+] Total de observações winsorizadas (v2): 9207 (corte condicional a não-nulos)
    Referência incondicional (com zeros):   11737 (induz à sobre-winsorização)
    Ganho de preservação de variação real:  2530 observações mantidas.

=== Ativos com Maior Preservação de Volatilidade Real (Diferença de Cortes) ===
            n_winsor_com_zeros  n_winsor_v2  reducao
ativo                                               
ACAO_SANB4                 436          159      277
ACAO_FICT3                 433          253      180
ACAO_VIVR3                 339          215      124
ACAO_RPMG3                 182           97       85
ACAO_LUPA3                 288          219       69
ACAO_HAGA4                 121           52       69
ACAO_PINE4                 137           78       59
ACAO_UNIP6                 101           42       59


## 5. Auditoria de Eventos Societários Corporativos

In [5]:
# Persiste as checagens manuais efetuadas na B3
verificacao = exportar_verificacao_corporativa(DIR_RETORNOS / "verificacao_eventos_corporativos.csv")
print("=== Registro e Parecer de Auditoria dos Extremos ===")
print(verificacao[["empresa","natureza_extremo","decisao"]].to_string())
print("\n[OK] Parecer salvo em verificacao_eventos_corporativos.csv")

=== Registro e Parecer de Auditoria dos Extremos ===
                            empresa                                                natureza_extremo                                                 decisao
ticker                                                                                                                                                     
FICT3   Fictor Alimentos (ex-ATOM3)                       Distress real + mudanca de base acionaria                                     Manter (ativo real)
GOLL54            Gol Linhas Aereas                       Artefato de evento (denominador ~centavo)  Manter; ideal ajustar preco a montante pelo grupamento
LUPA3                      Lupatech                                          Diluicao/distress real                                                  Manter
TELB4                 Telebras (PN)                        Artefato de split/grupamento + iliquidez                                  Manter; conferir datas
AMER3      

## 6. Testes Lógicos de Consistência Numérica

In [6]:
print("[+] Iniciando testes lógicos de sanidade matricial...\n")

assert ret_log_san.index.equals(ret_log.index), "Falha: Índice alterado."
assert ret_log_san.shape == ret_simples_san.shape == ret_log.shape, "Falha: Dimensões violadas."
print("      [OK] Estrutura dimensional isomórfica intacta.")

assert ret_log_san.isna().sum().sum() == 0 and ret_simples_san.isna().sum().sum() == 0, "Falha: Presença de NaNs."
assert np.isfinite(ret_log_san.values).all() and np.isfinite(ret_simples_san.values).all(), "Falha: Valores infinitos detectados."
print("      [OK] Matrizes livres de NaNs ou infinitos.")

assert np.allclose(ret_log_san.values, np.log1p(ret_simples_san.values), atol=1e-10), "Falha: Relação geométrica rompida."
print("      [OK] Relação matemática r_log = ln(1+R_simples) preservada com precisão de float64.")

mask_validos = ~rel["mad_zero"].values
resid = int((ret_simples_san.loc[:, mask_validos].abs() > LIMITE_IMPOSSIVEL).sum().sum())
print(f"      [OK] Retornos extremos remanescentes com volatilidade real (MAD>0): {resid}")

print("\n[OK] Validação de integridade aprovada in all asserções.")

[+] Iniciando testes lógicos de sanidade matricial...

      [OK] Estrutura dimensional isomórfica intacta.
      [OK] Matrizes livres de NaNs ou infinitos.
      [OK] Relação matemática r_log = ln(1+R_simples) preservada com precisão de float64.
      [OK] Retornos extremos remanescentes com volatilidade real (MAD>0): 0

[OK] Validação de integridade aprovada in all asserções.


## 7. Exportação de Dados Saneados

In [7]:
def _salvar(df, nome, ddir=DIR_RETORNOS):
    df.to_csv(ddir / f'{nome}.csv', index=True, date_format='%Y-%m-%d', float_format='%.8f')
    df.to_parquet(ddir / f'{nome}.parquet', engine='pyarrow')
    print(f'  • {nome}.csv e {nome}.parquet gerados.')

print('[+] Gravando retornos higienizados (winsorizados) em:', DIR_RETORNOS, '\n')
_salvar(ret_simples_san, 'retornos_simples_saneado')
_salvar(ret_log_san, 'retornos_log_saneado')

print('\n' + '='*50)
print(f'[OK] HIGIENIZAÇÃO TERMINADA: {len(cols_acoes)} ativos | {ret_simples_san.shape[0]} pregões')
print('='*50)

[+] Gravando retornos higienizados (winsorizados) em: C:\VSCodeWorkspace\1_TCC_Final\data\Retornos 



  • retornos_simples_saneado.csv e retornos_simples_saneado.parquet gerados.


  • retornos_log_saneado.csv e retornos_log_saneado.parquet gerados.

[OK] HIGIENIZAÇÃO TERMINADA: 118 ativos | 3966 pregões
